<a href="https://colab.research.google.com/github/mugalan/gsuite-based-attendance-monitoring/blob/main/QR_code_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#QR Code Generator

In [ ]:
import qrcode
import hashlib
import time
from IPython.display import display, Image, clear_output

# CONFIGURATION
WEB_APP_URL = "https://script.google.com/a/macros/eng.pdn.ac.lk/s/AKfycbzWp6lgKJxXYOGUU2AVaJwrSBnYk2M8WNqNtx_q_4DO6TkGbb9eL8Mgk--5a0n_rXRCGw/exec"
SECRET_PASSWORD = "ME526/ME2050"
# MUST match the Script password

time_step=30 #QR code refresh rate
def get_security_key():
    # Syncs with the 30-second window in the Apps Script
    time_slot = int(time.time() // time_step)
    raw_str = f"{SECRET_PASSWORD}{time_slot}"
    return hashlib.md5(raw_str.encode()).hexdigest()

try:
    n=0
    while True:
        clear_output(wait=True)
        n+=1

        # Create the secure link
        secure_key = get_security_key()

        # Add a timestamp to the end of the URL to prevent caching
        cache_buster = int(time.time() * 1000)
        secure_url = f"{WEB_APP_URL}?key={secure_key}&t={cache_buster}"

        # Generate QR
        qr = qrcode.make(secure_url)
        qr.save("secure_qr.png")

        print("🛡️ SECURE ATTENDANCE SYSTEM ACTIVE")
        print(f"Time: {time.strftime('%H:%M:%S')} | Key: {secure_key[:8]}...")
        print(f"Note: This QR expires every {time_step} seconds to prevent cheating. QR-code-{n}")

        display(Image("secure_qr.png"))
        time.sleep(time_step)
except KeyboardInterrupt:
    print("System Offline.")

#Google Sheet App Script

```
// Replace YOUR_SECRET_PASSWORD with any word you like
var SECRET_PASSWORD = "ME526/ME2050";

function doGet(e) {
  var userKey = e.parameter.key;
  var isValid = checkKey(userKey);

  var html = '<html><body style="font-family: sans-serif; text-align:center; padding:50px;">';
  
  if (!isValid) {
    html += '<h1 style="color:red;">❌ Invalid or Expired QR</h1>';
    html += '<p>Please scan the live QR code displayed in the classroom.</p>';
  } else {
    html += '<div id="main"><h2>Verify Attendance</h2>' +
            '<button id="btn" style="padding:15px 30px; font-size:18px; background:#1a73e8; color:white; border:none; border-radius:5px; cursor:pointer;" ' +
            'onclick="record()">Confirm My Identity</button></div>' +
            '<script>' +
            '  function record() {' +
            '    var btn = document.getElementById("btn");' +
            '    btn.disabled = true;' + // Disable button
            '    btn.innerHTML = "Please wait...";' + // Show feedback
            '    btn.style.backgroundColor = "#ccc";' + // Grey it out
            '    google.script.run.withSuccessHandler(done).withFailureHandler(fail).saveEmail("' + userKey + '");' +
            '  }' +
            '  function done(info) {' +
            '    document.getElementById("main").innerHTML = "<h1>✅ Success!</h1><p>Thanks, " + info.name + "!<br>Position: #" + info.count + "</p>";' +
            '  }' +
            '  function fail(err) {' +
            '    alert("Error: " + err.message + ". Please refresh the page.");' +
            '    document.getElementById("btn").disabled = false;' +
            '    document.getElementById("btn").innerHTML = "Confirm My Identity";' +
            '  }' +
            '</script>';
  }
  html += '</body></html>';
  return HtmlService.createHtmlOutput(html);
}

function checkKey(key) {
  var now = new Date();
  var timeSlot = Math.floor(now.getTime() / 30000); // 30-second window
  var rawStr = SECRET_PASSWORD + timeSlot;

  // 1. Compute digest with explicit US_ASCII charset
  var bytes = Utilities.computeDigest(Utilities.DigestAlgorithm.MD5, rawStr, Utilities.Charset.US_ASCII);
  
  // 2. Convert bytes to Hex string
  var hex = "";
  for (var i = 0; i < bytes.length; i++) {
    var b = bytes[i];
    if (b < 0) b += 256; // Handle signed byte
    var s = b.toString(16);
    if (s.length === 1) s = "0" + s; // Ensure 2-digit padding
    hex += s;
  }
  
  // 3. Optional: Add a "grace period" (Check previous time slot)
  var prevTimeSlot = timeSlot - 1;
  var prevRawStr = SECRET_PASSWORD + prevTimeSlot;
  var prevBytes = Utilities.computeDigest(Utilities.DigestAlgorithm.MD5, prevRawStr, Utilities.Charset.US_ASCII);
  var prevHex = "";
  for (var i = 0; i < prevBytes.length; i++) {
    var b = prevBytes[i];
    if (b < 0) b += 256;
    var s = b.toString(16);
    if (s.length === 1) s = "0" + s;
    prevHex += s;
  }

  // Return true if it matches current OR previous slot
  return (key === hex || key === prevHex);
}

function saveEmail(key) {
  if (!checkKey(key)) throw new Error("Security Timeout: Please re-scan");

  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var sheetName = Utilities.formatDate(new Date(), "GMT+5:30", "dd-MM-yyyy");
  var sheet = ss.getSheetByName(sheetName) || ss.insertSheet(sheetName, 0);
  
  var user = Session.getActiveUser();
  var email = user.getEmail();
  var name = email.split('@')[0];
  var timeString = Utilities.formatDate(new Date(), "GMT+5:30", "HH:mm:ss");

  sheet.appendRow([timeString, email, name]);
  return { name: name, count: (sheet.getLastRow()), time: timeString };
}
```